In [50]:
import scanpy as sc
import numpy as np
import scipy.sparse as sp

# ── 1. Load raw data ──
adata = sc.read_h5ad(
    "/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/full_perturbseq_datasets/K562_essential_norm.h5ad"
)
print("Shape:", adata.shape)
print("X type:", type(adata.X))
print("X dtype:", adata.X.dtype)
print()

# ── 2. Check for problems BEFORE any preprocessing ──
X = adata.X
if sp.issparse(X):
    data = X.data  # only the stored (non-zero) entries
    print("Sparse format:", X.format)
else:
    data = X.ravel()

print(f"NaN  in raw X.data: {np.isnan(data).sum()}")
print(f"Inf  in raw X.data: {np.isinf(data).sum()}")
print(f"Neg  in raw X.data: {(data < 0).sum()}")
print(f"Min  of raw X.data: {data.min()}")
print(f"Max  of raw X.data: {data.max()}")
print(f"Mean of raw X.data: {data.mean()}")
print()

# ── 3. Per-cell totals ──
cell_totals = np.array(X.sum(axis=1)).flatten()
print(f"Cells with zero total counts: {(cell_totals == 0).sum()}")
print(f"Cells with NaN  total counts: {np.isnan(cell_totals).sum()}")
print(f"Cells with Inf  total counts: {np.isinf(cell_totals).sum()}")
print(f"Cell total min: {cell_totals.min():.4f}, max: {cell_totals.max():.4f}, mean: {cell_totals.mean():.4f}")
print()

# ── 4. Per-gene totals ──
gene_totals = np.array(X.sum(axis=0)).flatten()
print(f"Genes with zero total counts: {(gene_totals == 0).sum()}")
print(f"Genes with NaN  total counts: {np.isnan(gene_totals).sum()}")
print()

# ── 5. Check if data was ALREADY log-transformed (heuristic: max < ~20 suggests log-space) ──
print(f"Max value in X: {data.max():.4f}")
if data.max() < 20:
    print("⚠️  Max value is small — data may ALREADY be log-normalized!")
elif data.max() > 1e5:
    print("Data looks like raw counts (large max).")
else:
    print("Data might be normalized but not log-transformed.")
print()

# ── 6. Check .uns for prior normalization markers ──
print("adata.uns keys:", list(adata.uns.keys()))
if "log1p" in adata.uns:
    print("⚠️  'log1p' key found in .uns — data was likely ALREADY log1p-transformed!")
    print("    adata.uns['log1p'] =", adata.uns["log1p"])
print()

# ── 7. Check .layers for a raw layer ──
print("adata.layers keys:", list(adata.layers.keys()))
if "counts" in adata.layers:
    print("A 'counts' layer exists — you may want to use adata.layers['counts'] as raw.")
print()

# ── 8. Check .raw ──
print("adata.raw is not None:", adata.raw is not None)

Shape: (310385, 8563)
X type: <class 'numpy.ndarray'>
X dtype: float32

NaN  in raw X.data: 0
Inf  in raw X.data: 0
Neg  in raw X.data: 1814491761
Min  of raw X.data: -6.354443550109863
Max  of raw X.data: 1909.4228515625
Mean of raw X.data: 0.01686471328139305

Cells with zero total counts: 0
Cells with NaN  total counts: 0
Cells with Inf  total counts: 0
Cell total min: -3232.3115, max: 5258.8638, mean: 144.4120

Genes with zero total counts: 0
Genes with NaN  total counts: 0

Max value in X: 1909.4229
Data might be normalized but not log-transformed.

adata.uns keys: []

adata.layers keys: []

adata.raw is not None: False


In [51]:
adata

AnnData object with n_obs × n_vars = 310385 × 8563
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'

In [2]:
import scanpy as sc


In [23]:
adata = sc.read_h5ad('/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/morph_depmap_prior/K562_gwps_normalized_hvg_gene_names.h5ad', backed=None)
adata


AnnData object with n_obs × n_vars = 1990505 × 5000
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'n_counts', 'batch', 'cell_barcode'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'

In [27]:
adata.uns['log1p']

{}

In [25]:
assert adata.X is not None, "X is None after load"

AssertionError: X is None after load

In [9]:
adata.obs['cell_line']

0         K562
1         K562
2         K562
3         K562
4         K562
          ... 
111117    K562
111118    K562
111119    K562
111120    K562
111121    K562
Name: cell_line, Length: 111122, dtype: category
Categories (1, object): ['K562']

In [28]:
import scanpy as sc

heldout = [
    "HDAC7", "CNOT3", "POLR1B", "RPL30", "RPL17", "RPS27", "PHB", "ZC3H13",
    "LSM6", "NACA", "YTHDC1", "EIF2S1", "EXOSC2", "RPS5", "RPS15A", "PRIM2",
    "SMG5", "EIF4A3"
]

adata = sc.read_h5ad("/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/full_perturbseq_datasets/K562_gwps_norm.h5ad")
counts = adata.obs["gene"].value_counts()

for g in heldout:
    n = counts.get(g, 0)
    print(f"{g}: {n} cells")
print("---")
print("Total cells (heldout):", sum(counts.get(g, 0) for g in heldout))
print("All perturbation targets in adata:", set(adata.obs["gene"].unique()) - {"non-targeting"})

HDAC7: 50 cells
CNOT3: 232 cells
POLR1B: 108 cells
RPL30: 161 cells
RPL17: 32 cells
RPS27: 87 cells
PHB: 112 cells
ZC3H13: 148 cells
LSM6: 83 cells
NACA: 160 cells
YTHDC1: 137 cells
EIF2S1: 89 cells
EXOSC2: 161 cells
RPS5: 41 cells
RPS15A: 175 cells
PRIM2: 65 cells
SMG5: 159 cells
EIF4A3: 24 cells
---
Total cells (heldout): 2024
All perturbation targets in adata: {'STX12', 'ASAH2B', 'PTK7', 'ZNF880', 'XPR1', 'FERD3L', 'ZNF91', 'ZNF276', 'ZNF681', 'MRPS30', 'TRIM26', 'NHEJ1', 'ALMS1', 'KLHL11', 'ALX4', 'ZNF721', 'CNST', 'ETV4', 'FSTL3', 'ABCB10', 'SNRPD2', 'POLR2D', 'RPL22L1', 'CHCHD4', 'POFUT2', 'UBE3A', 'HIST1H2AL', 'U2SURP', 'ZNF836', 'QPRT', 'HAUS2', 'PWWP2B', 'EEA1', 'MMGT1', 'USP53', 'POMT1', 'KLF16', 'SAR1B', 'ISG15', 'PIK3R1', 'NFS1', 'SPPL2A', 'MCFD2', 'C9orf16', 'PRDM16', 'SH3BP5L', 'GTF2A2', 'EIF5B', 'BNIP3L', 'PGAM1', 'BAZ2A', 'HIRIP3', 'UBE4A', 'MMADHC', 'ANKLE2', 'IBA57', 'SETD6', 'RANBP3', 'PDCD2', 'CLPB', 'KCNMB3', 'INHBE', 'DNAJB4', 'TAF1C', 'TTLL3', 'MED19', 'GGA2', 'N

In [ ]:
import scanpy as sc

heldout = [
    "HDAC7", "CNOT3", "POLR1B", "RPL30", "RPL17", "RPS27", "PHB", "ZC3H13",
    "LSM6", "NACA", "YTHDC1", "EIF2S1", "EXOSC2", "RPS5", "RPS15A", "PRIM2",
    "SMG5", "EIF4A3"
]

adata = sc.read_h5ad("/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/full_perturbseq_datasets/K562_essential_norm.h5ad")
counts = adata.obs["gene"].value_counts()

for g in heldout:
    n = counts.get(g, 0)
    print(f"{g}: {n} cells")
print("---")
print("Total cells (heldout):", sum(counts.get(g, 0) for g in heldout))
print("All perturbation targets in adata:", set(adata.obs["gene"].unique()) - {"non-targeting"})

HDAC7: 171 cells
CNOT3: 332 cells
POLR1B: 181 cells
RPL30: 365 cells
RPL17: 149 cells
RPS27: 174 cells
PHB: 256 cells
ZC3H13: 0 cells
LSM6: 461 cells
NACA: 354 cells
YTHDC1: 237 cells
EIF2S1: 243 cells
EXOSC2: 226 cells
RPS5: 144 cells
RPS15A: 314 cells
PRIM2: 156 cells
SMG5: 83 cells
EIF4A3: 7 cells
---
Total cells (heldout): 3853
All perturbation targets in adata: {'GTF2H2', 'SAMM50', 'CRLS1', 'EXOSC8', 'UBTF', 'HAUS8', 'MCMBP', 'COG1', 'KLHL11', 'MRPL50', 'ABCB10', 'SNRPD2', 'POLR2D', 'TUBGCP4', 'CHCHD4', 'SPCS2', 'NXF1', 'DNLZ', 'NACA', 'OSGEP', 'U2SURP', 'TEFM', 'MED27', 'MMGT1', 'TADA2A', 'NUS1', 'DDX1', 'UQCRB', 'NFS1', 'CMTR1', 'WDR54', 'PSMB3', 'RBM28', 'C9orf16', 'ARL4D', 'RPS7', 'COQ4', 'IFITM3', 'GTF2A2', 'TFB2M', 'RBMX', 'VPS28', 'MPHOSPH10', 'PGAM1', 'OTOP1', 'ANKLE2', 'DMAP1', 'TELO2', 'PDCD2', 'NBAS', 'UTP6', 'DNM2', 'CRCP', 'TSSK3', 'TAF1C', 'MED19', 'CRKL', 'YTHDC1', 'OIP5', 'DDX54', 'SMARCB1', 'RPL3', 'OTX1', 'HMGB1', 'MBTPS2', 'NAF1', 'CDK7', 'CWF19L2', 'MRPS9', 'PO

In [41]:
cell_totals = np.array(adata.X.sum(axis=1)).flatten()
print("Cells with zero total counts:", (cell_totals == 0).sum())

Cells with zero total counts: 0


In [45]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

/lotterlab/users/riccardo/ML_BIO/Bio_FMs/RNA/!SCFM_meta/.venv/lib/python3.12/site-packages/scanpy/preprocessing/_simple.py:391: RuntimeWarning: divide by zero encountered in log1p
  np.log1p(x, out=x)
/lotterlab/users/riccardo/ML_BIO/Bio_FMs/RNA/!SCFM_meta/.venv/lib/python3.12/site-packages/scanpy/preprocessing/_simple.py:391: RuntimeWarning: invalid value encountered in log1p
  np.log1p(x, out=x)


In [49]:
adata.X.max()

nan

In [40]:
from scipy.sparse import issparse
import numpy as np
X = adata.X.toarray() if issparse(adata.X) else adata.X
np.isnan(X).any()

False

In [ ]:
hvg_cache_dir = os.path.join(base_dir, 'data', 'hvg_cache')
adata_stem = os.path.splitext(os.path.basename(adata_path))[0]
hvg_cache_path = os.path.join(hvg_cache_dir, f'{adata_stem}_n{n_top_genes}.pkl')

if os.path.isfile(hvg_cache_path):
    with open(hvg_cache_path, 'rb') as f:
        hvg_genes = pickle.load(f)
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    _sanitize_x(adata)
    genes_in_adata = [g for g in hvg_genes if g in adata.var_names]
    adata = adata[:, genes_in_adata].copy()
    print('Loaded HVG from cache: %s (%s genes)' % (hvg_cache_path, adata.shape[1]))
else:
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    _sanitize_x(adata)
    print('Identifying HVGs...')
    time_start = time.time()
    sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, subset=True)
    print('Computed HVG and subset: n_top_genes=%s, %s genes' % (n_top_genes, adata.shape[1]))
    print('Time taken: ', time.time() - time_start, ' seconds')
    os.makedirs(hvg_cache_dir, exist_ok=True)
    with open(hvg_cache_path, 'wb') as f:
        pickle.dump(adata.var_names.tolist(), f, protocol=pickle.HIGHEST_PROTOCOL)
    print('Saved HVG list to: ', hvg_cache_path)


In [32]:
adata = sc.read_h5ad("/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/morph_depmap_prior/Norman2019_normalized_hvg.h5ad")

/lotterlab/users/riccardo/ML_BIO/Bio_FMs/RNA/!SCFM_meta/.venv/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [34]:
adata.X.isna()

AttributeError: 'numpy.ndarray' object has no attribute 'isna'

In [13]:
hvg_genes = adata.var_names.tolist()
# Optional: symbol column if present
if "symbol" in adata.var.columns:
    hvg_symbols = adata.var["symbol"].tolist()
# Confirm all are marked HVG
assert adata.var["highly_variable"].all()

In [16]:
import scanpy as sc

# Norman HVG gene set (var_names = Ensembl IDs in that file)
adata_norman = sc.read_h5ad("/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/morph_depmap_prior/Norman2019_normalized_hvg.h5ad")
hvg_genes = adata_norman.var_names.tolist()

# Essential adata
adata = sc.read_h5ad("/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/full_perturbseq_datasets/K562_essential_norm.h5ad")

# Genes that exist in both (same ID space: check if essential uses Ensembl or symbols)
genes_in_both = [g for g in hvg_genes if g in adata.var_names]
print(f"Overlap: {len(genes_in_both)} of {len(hvg_genes)} Norman HVG genes in essential")

# Subset essential to those genes only (all cells kept)
adata_subset = adata[:, genes_in_both].copy()
# Optional: mark as HVG for downstream
adata_subset.var["highly_variable"] = True

/lotterlab/users/riccardo/ML_BIO/Bio_FMs/RNA/!SCFM_meta/.venv/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Overlap: 818 of 5044 Norman HVG genes in essential


In [17]:
print("Norman var_names sample:", adata_norman.var_names[:5].tolist())
print("Norman var.columns:", adata_norman.var.columns.tolist())
if "symbol" in adata_norman.var.columns:
    print("Norman symbol sample:", adata_norman.var["symbol"].head().tolist())

# Essential
print("Essential var_names sample:", adata.var_names[:5].tolist())
print("Essential var.columns:", adata.var.columns.tolist())

Norman var_names sample: ['ENSG00000239945', 'ENSG00000223764', 'ENSG00000187634', 'ENSG00000187583', 'ENSG00000188290']
Norman var.columns: ['ensemble_id', 'ncounts', 'ncells', 'symbol', 'highly_variable', 'means', 'dispersions', 'dispersions_norm']
Norman symbol sample: ['RP11-34P13.8', 'RP11-54O7.3', 'SAMD11', 'PLEKHN1', 'HES4']
Essential var_names sample: ['ENSG00000237491', 'ENSG00000228794', 'ENSG00000188976', 'ENSG00000187961', 'ENSG00000188290']
Essential var.columns: ['gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano']


In [18]:
# How many genes in essential total?
print("Essential n_vars:", adata.shape[1])
# How many of Norman's HVG Ensembl IDs exist in essential?
norman_hvg_ens = set(adata_norman.var_names)
in_essential = [g for g in adata.var_names if g in norman_hvg_ens]
print("Norman HVG genes present in essential:", len(in_essential))

Essential n_vars: 8563
Norman HVG genes present in essential: 818


In [5]:
import scanpy as sc
p = '/lotterlab/datasets/VCC/DATASETS/SC_PerturbSeq_datasets/morph_depmap_prior/K562_gwps_normalized_hvg_gene_names.h5ad'
adata = sc.read_h5ad(p)
print('shape (n_cells, n_genes):', adata.shape)
print('var columns:', list(adata.var.columns))
print('uns keys:', list(adata.uns.keys())[:20])
if 'highly_variable' in adata.var.columns:
    print('highly_variable in var: True, n HVG:', adata.var['highly_variable'].sum())
else:
    print('highly_variable in var: False')
print('var_names sample (first 5):', list(adata.var_names[:5]))

shape (n_cells, n_genes): (1990505, 5000)
var columns: ['gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm']
uns keys: ['hvg', 'log1p']
highly_variable in var: True, n HVG: 5000
var_names sample (first 5): ['ENSG00000237491', 'ENSG00000188290', 'ENSG00000187608', 'ENSG00000078808', 'ENSG00000131584']
